In [1]:
import pygame
import numpy as np
from env.config import FPS, PLAYER_MAX_ACCELERATION, PLAYER_MAX_ANGULAR_ACCELERATION, PLAYER_MAX_KICKING_FORCE
from env.environment import FootballEnv
from env.engine import Object, Player, Ball, distance


pygame 2.6.1 (SDL 2.28.4, Python 3.11.3)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
# def main():
#     # Initialize Pygame
#     pygame.init()

#     # Create environment with exactly 1 player per team
#     env = FootballEnv(team_size=1)
#     state = env.reset()

#     # Define positions for the two players
#     center_x = env.dimensions.stadium_length / 4
#     center_y = env.dimensions.stadium_width / 2
#     distance = 10  # Distance between players
    
#     # Position players facing each other
#     positions = [
#         (center_x - distance/2, center_y),  # Team 0 player (left)
#         (center_x + distance/2, center_y)   # Team 1 player (right)
#     ]

#     # Set player positions and orientations
#     for i, player in enumerate(env.players):
#         player.object.position = positions[i]
#         player.object.orientation = 0 if i == 0 else np.pi
#         player.object.velocity = (0, 0)
#         player.object.angular_velocity = 0.0

#     # Place ball at first player position
#     env.ball.object.position = positions[0]
#     env.ball.object.velocity = (0, 0)

#     # Initialize passing sequence
#     current_idx = 0
#     passing_stage = "ready"  # States: ready, passing, receiving
#     wait_time = 0
    
#     # Kicking parameters
#     pass_power = 25.0  # Increased kicking force to overcome friction
#     max_pass_time = 120  # Allow more time for the pass to arrive

#     running = True
#     clock = pygame.time.Clock()
#     pass_timer = 0

#     # Render once to initialize
#     env.render()
    
#     while running:
#         for event in pygame.event.get():
#             if event.type == pygame.QUIT:
#                 running = False

#         # Default no-op actions (no movement, no kick)
#         actions = [(0, 0, 0, 0) for _ in range(env.num_players)]
        
#         # State machine for passing
#         if passing_stage == "ready":
#             current_player = env.players[current_idx]
#             next_idx = 1 - current_idx
#             next_player = env.players[next_idx]

#             # Calculate direction to target player
#             dx = next_player.object.position[0] - current_player.object.position[0]
#             dy = next_player.object.position[1] - current_player.object.position[1]
#             angle_to_target = np.arctan2(dy, dx)
            
#             # Set kicking action for current player (acceleration=0, angular=0, kick power, kick angle)
#             actions[current_idx] = (0, 0, pass_power, angle_to_target - current_player.object.orientation)
            
#             print(f"Player {current_idx} attempting pass to Player {next_idx}")
#             passing_stage = "passing"
#             pass_timer = 0

#         elif passing_stage == "passing":
#             pass_timer += 1
            
#             # Check if ball has reached target player
#             next_idx = 1 - current_idx
#             next_player = env.players[next_idx]
#             dist_to_next = np.hypot(
#                 next_player.object.position[0] - env.ball.object.position[0],
#                 next_player.object.position[1] - env.ball.object.position[1]
#             )
            
#             sum_radii = env.dimensions.player_radius + env.dimensions.ball_radius
#             if dist_to_next <= sum_radii + 1.0:  # Small buffer for capture
#                 passing_stage = "waiting"
#                 wait_time = 30
#                 current_idx = next_idx
#                 print(f"Ball received by Player {next_idx}")
            
#             # Reset if pass takes too long
#             if pass_timer > max_pass_time:
#                 passing_stage = "waiting"
#                 wait_time = 30
#                 current_idx = next_idx
#                 print("Pass timed out")

#         elif passing_stage == "waiting":
#             wait_time -= 1
#             if wait_time <= 0:
#                 passing_stage = "ready"

#         # Step the environment with calculated actions
#         state, rewards, done, info = env.step(actions)
        
#         env.render()
#         clock.tick(60)

#     pygame.quit()

In [3]:
# main()

In [ ]:
def main1():
    pygame.init()
    env = FootballEnv(team_size=2)
    state = env.reset()

    center_x = env.dimensions.stadium_length / 2
    center_y = env.dimensions.stadium_width / 2
    distance = 8  # Reduced distance

    positions = [
        (center_x - distance, center_y),
        (center_x, center_y + distance),
        (center_x + distance, center_y),
        (center_x, center_y - distance)
    ]

    # Set orientations PROPERLY
    for i, player in enumerate(env.players):
        player.object.position = positions[i]
        next_i = (i + 1) % 4
        dx = positions[next_i][0] - positions[i][0]
        dy = positions[next_i][1] - positions[i][1]
        player.object.orientation = np.arctan2(dy, dx) % (2 * np.pi)
        player.object.velocity = (0, 0)
        player.object.angular_velocity = 0.0

    env.ball.object.position = positions[0]
    env.ball.object.velocity = (0, 0)

    current_idx = 0
    passing_stage = "ready"
    wait_time = 0
    pass_power = 35.0  # Increased power as direct velocity
    max_pass_time = 60  # Shorter timeout

    running = True
    clock = pygame.time.Clock()
    pass_timer = 0

    env.render()
    
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

        actions = [(0, 0, 0, 0) for _ in range(env.num_players)]
        
        if passing_stage == "ready":
            next_idx = (current_idx + 1) % 4
            current_player = env.players[current_idx]
            
            # Calculate RELATIVE kick angle (should be near zero if oriented correctly)
            dx = positions[next_idx][0] - current_player.object.position[0]
            dy = positions[next_idx][1] - current_player.object.position[1]
            target_angle = np.arctan2(dy, dx)
            kick_angle = target_angle - current_player.object.orientation
            
            actions[current_idx] = (0, 0, pass_power, kick_angle)
            print(f"Player {current_idx} passing to Player {next_idx}")
            passing_stage = "passing"
            pass_timer = 0

        elif passing_stage == "passing":
            pass_timer += 1
            next_idx = (current_idx + 1) % 4
            target_player = env.players[next_idx]
            
            dist_to_target = np.hypot(
                target_player.object.position[0] - env.ball.object.position[0],
                target_player.object.position[1] - env.ball.object.position[1]
            )
            
            sum_radii = env.dimensions.player_radius + env.dimensions.ball_radius
            if dist_to_target <= sum_radii + 1.5:
                passing_stage = "waiting"
                wait_time = 15  # Reduced waiting
                current_idx = next_idx
                print(f"Ball received by Player {next_idx}")
            
            if pass_timer > max_pass_time:
                passing_stage = "waiting"
                wait_time = 15
                current_idx = next_idx
                print("Pass timed out")

        elif passing_stage == "waiting":
            wait_time -= 1
            if wait_time <= 0:
                passing_stage = "ready"

        state, rewards, done, info = env.step(actions)
        env.render()
        clock.tick(60)

    pygame.quit()

In [27]:
main1()

Player 0 passing to Player 1
Pass timed out
Player 1 passing to Player 2
Pass timed out
Player 2 passing to Player 3
Pass timed out
Player 3 passing to Player 0
Ball received by Player 0
Player 0 passing to Player 1
Pass timed out
Player 1 passing to Player 2


In [28]:
# def fighting_for_ball():
#     # Initialize Pygame
#     pygame.init()

#     # Create environment with exactly 1 player per team
#     env = FootballEnv(team_size=1)
#     state = env.reset()

#     # Define positions for the two players - start them farther apart
#     center_x = env.dimensions.stadium_length / 2
#     center_y = env.dimensions.stadium_width / 2
#     distance = 50  # Increased distance between players
    
#     # Position players with some separation
#     positions = [
#         (center_x - distance/2, center_y - 10),  # Team 0 player (left)
#         (center_x + distance/2, center_y + 10)   # Team 1 player (right)
#     ]

#     # Set player positions and orientations
#     for i, player in enumerate(env.players):
#         player.object.position = positions[i]
#         # Orient players to face center
#         player.object.orientation = 0 if i == 0 else np.pi
#         player.object.velocity = (0, 0)
#         player.object.angular_velocity = 0.0

#     # Place ball at center, not at player position
#     env.ball.object.position = (center_x, center_y)
#     env.ball.object.velocity = (0, 0)

#     # Parameters for player movement
#     player_speed = 0.7
#     turning_speed = 0.25
#     possession_threshold = env.dimensions.player_radius + env.dimensions.ball_radius + 0.2
    
#     # Track which player has possession
#     ball_possession = None
#     possession_cooldown = 0
#     steal_cooldown = 0
    
#     running = True
#     clock = pygame.time.Clock()

#     # Render once to initialize
#     env.render()
    
#     while running:
#         for event in pygame.event.get():
#             if event.type == pygame.QUIT:
#                 running = False

#         # Default no-op actions - will be updated for each player
#         actions = [(0, 0, 0, 0) for _ in range(env.num_players)]
        
#         # Current ball position
#         ball_pos = env.ball.object.position
        
#         # Update possession status
#         if possession_cooldown > 0:
#             possession_cooldown -= 1
        
#         if steal_cooldown > 0:
#             steal_cooldown -= 1
            
#         # Check if any player has possession
#         for i, player in enumerate(env.players):
#             player_pos = player.object.position
#             dist_to_ball = np.hypot(
#                 player_pos[0] - ball_pos[0],
#                 player_pos[1] - ball_pos[1]
#             )
            
#             # Give possession to player if close enough and no cooldown
#             if dist_to_ball < possession_threshold and possession_cooldown == 0:
#                 # Allow stealing only if steal cooldown is over
#                 if ball_possession is None or ball_possession != i and steal_cooldown == 0:
#                     if ball_possession is not None and ball_possession != i:
#                         print(f"Player {i} stole the ball from Player {ball_possession}")
#                         steal_cooldown = 30  # Set cooldown for next steal attempt
#                     else:
#                         print(f"Player {i} gained possession")
                    
#                     ball_possession = i
#                     possession_cooldown = 5  # Small cooldown to prevent flickering possession
        
#         # Process each player's actions based on ball position
#         for i, player in enumerate(env.players):
#             player_pos = player.object.position
#             player_orient = player.object.orientation
            
#             # Calculate direction to ball
#             dx = ball_pos[0] - player_pos[0]
#             dy = ball_pos[1] - player_pos[1]
#             angle_to_ball = np.arctan2(dy, dx)
            
#             # Normalize to [-π, π]
#             while player_orient > np.pi:
#                 player_orient -= 2 * np.pi
#             while player_orient < -np.pi:
#                 player_orient += 2 * np.pi
                
#             # Calculate angle difference
#             angle_diff = angle_to_ball - player_orient
#             while angle_diff > np.pi:
#                 angle_diff -= 2 * np.pi
#             while angle_diff < -np.pi:
#                 angle_diff += 2 * np.pi
            
#             # Player behavior depends on possession
#             if ball_possession == i:
#                 # Player has the ball - move in a circle to evade other player
#                 time_factor = pygame.time.get_ticks() / 1000.0
#                 # Create a circular movement pattern
#                 actions[i] = (
#                     player_speed * 0.7,  # Move forward slightly slower when in possession
#                     turning_speed * np.sin(time_factor),  # Turn based on time
#                     0,  # No sliding
#                     0   # No action button
#                 )
                
#                 # Move ball with the player (slightly in front)
#                 offset_x = np.cos(player.object.orientation) * (env.dimensions.player_radius + env.dimensions.ball_radius + 0.1)
#                 offset_y = np.sin(player.object.orientation) * (env.dimensions.player_radius + env.dimensions.ball_radius + 0.1)
                
#                 env.ball.object.position = (
#                     player_pos[0] + offset_x,
#                     player_pos[1] + offset_y
#                 )
#                 env.ball.object.velocity = player.object.velocity
                
#             else:
#                 # Player doesn't have the ball - chase it
#                 # First turn toward ball
#                 if abs(angle_diff) > 0.1:
#                     turn_direction = np.sign(angle_diff) * turning_speed
#                     actions[i] = (0, turn_direction, 0, 0)
#                 else:
#                     # Then move toward ball
#                     actions[i] = (player_speed, 0, 0, 0)
        
#         # If no one has possession, add some friction to the ball
#         if ball_possession is None:
#             env.ball.object.velocity = (
#                 env.ball.object.velocity[0] * 0.98,
#                 env.ball.object.velocity[1] * 0.98
#             )
            
#             # If ball is moving very slowly, stop it completely
#             if np.hypot(env.ball.object.velocity[0], env.ball.object.velocity[1]) < 0.01:
#                 env.ball.object.velocity = (0, 0)
        
#         # Handle ball and player going out of bounds
#         stadium_length = env.dimensions.stadium_length
#         stadium_width = env.dimensions.stadium_width
#         margin = 2.0  # Keep away from the edges
        
#         # Keep ball in bounds
#         ball_x, ball_y = env.ball.object.position
#         if ball_x < margin:
#             env.ball.object.position = (margin, ball_y)
#             env.ball.object.velocity = (abs(env.ball.object.velocity[0]), env.ball.object.velocity[1])
#         elif ball_x > stadium_length - margin:
#             env.ball.object.position = (stadium_length - margin, ball_y)
#             env.ball.object.velocity = (-abs(env.ball.object.velocity[0]), env.ball.object.velocity[1])
            
#         if ball_y < margin:
#             env.ball.object.position = (ball_x, margin)
#             env.ball.object.velocity = (env.ball.object.velocity[0], abs(env.ball.object.velocity[1]))
#         elif ball_y > stadium_width - margin:
#             env.ball.object.position = (ball_x, stadium_width - margin)
#             env.ball.object.velocity = (env.ball.object.velocity[0], -abs(env.ball.object.velocity[1]))
        
#         # Step the environment
#         state, rewards, done, info = env.step(actions)
        
#         env.render()
#         clock.tick(60)  # 60 FPS for smooth animation
        
#     pygame.quit()

In [29]:
# fighting_for_ball()